# Lab 4 — Build the Audience, Then Buy It
## Name: Ihor Lemeshko | Group: KM-42

In [15]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

## Task 1.0 — Clean the survey file

In [16]:
import numpy as np

display(survey_layout)

# Identify adult panel members (R and L records, age 18+)
# In viewer_attributes: 101 = R, 102 = L, 300 = Age in years
rl_ids = set(viewer_attributes.loc[viewer_attributes["attr_id"].isin([101, 102]), "respondent_id"])
ages = viewer_attributes.loc[viewer_attributes["attr_id"] == 300, ["respondent_id", "value"]].copy()
adult_ids = set(ages.loc[(ages["respondent_id"].isin(rl_ids)) & (ages["value"] >= 18), "respondent_id"])

raw_rows = len(survey)
df_s = survey.copy()

df_s["RESPID_clean"] = df_s["RESPID"].str.strip()
df_s["KEYWORD_clean"] = df_s["KEYWORD"].str.strip()
df_s["VALUE_clean"] = df_s["VALUE"].str.strip().str.lower()

bad_records = []

# Check unknown KEYWORD (not in RECTYPE == "B" of layout)
valid_keywords = set(survey_layout.loc[survey_layout["RECTYPE"] == "B", "KEYWORD"].str.strip())
mask_bad_kw = ~df_s["KEYWORD_clean"].isin(valid_keywords)
bad_records.append({
    "kind_of_bad_record": "Unknown KEYWORD (not in layout)",
    "rows": int(mask_bad_kw.sum()),
    "action": "Dropped",
})
df_s = df_s[~mask_bad_kw].copy()

# Check non-adult or non-panel RESPID (non-numeric, H/S, <18, or unknown)
df_s["resp_num"] = pd.to_numeric(df_s["RESPID_clean"], errors="coerce")
mask_bad_id = ~df_s["resp_num"].isin(adult_ids)
bad_records.append({
    "kind_of_bad_record": "Non-adult or non-panel RESPID (H, S, <18, or unknown)",
    "rows": int(mask_bad_id.sum()),
    "action": "Dropped",
})
df_s = df_s[~mask_bad_id].copy()
df_s["respondent_id"] = df_s["resp_num"].astype(int)

# Check VALUE is not "yes"
mask_bad_val = df_s["VALUE_clean"] != "yes"
bad_records.append({
    "kind_of_bad_record": "VALUE is not \"yes\"",
    "rows": int(mask_bad_val.sum()),
    "action": "Dropped",
})
df_s = df_s[~mask_bad_val].copy()

# Check duplicate (respondent_id, KEYWORD) rows
mask_dup = df_s.duplicated(subset=["respondent_id", "KEYWORD_clean"], keep="first")
bad_records.append({
    "kind_of_bad_record": "Duplicate (RESPID, KEYWORD) row",
    "rows": int(mask_dup.sum()),
    "action": "Dropped duplicates",
})
df_s = df_s[~mask_dup].copy()

# Identify keywords for pair counts, Any, and brands from layout
b_rows = survey_layout[survey_layout["RECTYPE"] == "B"].copy()
print(b_rows[["KEYWORD", "Text", "Variable", "SAMPLE"]].to_string(index=False))

kw_any = b_rows.loc[
    b_rows["Text"].str.contains("Any|bought", case=False, na=False)
    & ~b_rows["Text"].str.contains("pair|1|2|3", case=False, na=False),
    "KEYWORD",
].iloc[0]

pair_rows = b_rows[b_rows["Text"].str.contains("pair", case=False, na=False)].copy()
pair_rows["pair_num"] = pair_rows["Text"].str.extract(r"(\d+)").astype(int)
pair_rank_map = dict(zip(pair_rows["KEYWORD"], pair_rows["pair_num"]))
rank_to_kw = dict(zip(pair_rows["pair_num"], pair_rows["KEYWORD"]))
brand_kws = [k for k in valid_keywords if k not in pair_rank_map and k != kw_any]

# Check people with two or more pair count answers -> keep the higher one
df_pairs = df_s[df_s["KEYWORD_clean"].isin(pair_rank_map)].copy()
df_pairs["pair_num"] = df_pairs["KEYWORD_clean"].map(pair_rank_map)
multi_pair_people = df_pairs.groupby("respondent_id").filter(lambda g: len(g) > 1)
n_two_pairs = multi_pair_people["respondent_id"].nunique()
rows_extra_pairs = len(multi_pair_people) - n_two_pairs
bad_records.append({
    "kind_of_bad_record": "Multiple pair counts for same person",
    "rows": int(rows_extra_pairs),
    "action": "Kept highest pair count",
})

max_pair_per_person = df_pairs.groupby("respondent_id")["pair_num"].max()
df_s = df_s[~df_s["KEYWORD_clean"].isin(pair_rank_map)].copy()
best_pair_df = pd.DataFrame({
    "respondent_id": max_pair_per_person.index,
    "KEYWORD_clean": max_pair_per_person.map(rank_to_kw).values,
})
df_s = pd.concat([df_s[["respondent_id", "KEYWORD_clean"]], best_pair_df], ignore_index=True)

# Check person has pair count or brand, but no "Any" row -> add "Any"
people_with_pair_or_brand = set(df_s.loc[df_s["KEYWORD_clean"] != kw_any, "respondent_id"])
people_with_any = set(df_s.loc[df_s["KEYWORD_clean"] == kw_any, "respondent_id"])
missing_any_ids = sorted(people_with_pair_or_brand - people_with_any)
bad_records.append({
    "kind_of_bad_record": "Has pairs/brand but missing \"Any\" row",
    "rows": len(missing_any_ids),
    "action": "Imputed \"Any\" = yes",
})

if missing_any_ids:
    df_s = pd.concat([
        df_s,
        pd.DataFrame({"respondent_id": missing_any_ids, "KEYWORD_clean": kw_any}),
    ], ignore_index=True)

# Collapse to one row per adult panel member
adults_df = pd.DataFrame({"respondent_id": sorted(adult_ids)})
piv = (
    pd.crosstab(df_s["respondent_id"], df_s["KEYWORD_clean"])
    .reindex(adults_df["respondent_id"], fill_value=0)
)

any_vec = piv[kw_any].to_numpy() if kw_any in piv.columns else np.zeros(len(adults_df), dtype=int)
adults_df["buyer"] = np.where(any_vec > 0, "yes", "no")

adults_df["pairs"] = 0
for p_num, p_kw in rank_to_kw.items():
    if p_kw in piv.columns:
        mask = (piv[p_kw] > 0).to_numpy()
        adults_df.loc[mask, "pairs"] = p_num

for b_kw in brand_kws:
    b_vec = piv[b_kw].to_numpy() if b_kw in piv.columns else np.zeros(len(adults_df), dtype=int)
    adults_df[b_kw] = np.where(b_vec > 0, "yes", "no")

print("Survey data rows:", raw_rows)
print("Clean yes answers:", len(df_s))
print("Adult panel members:", len(adults_df))
print("Non-buyers:", int((adults_df["buyer"] == "no").sum()))
print("People with two pair counts:", n_two_pairs)

display(pd.DataFrame(bad_records))

# Check counts against layout SAMPLE
check_df = b_rows[["KEYWORD", "Text", "SAMPLE"]].copy()
check_df["SAMPLE"] = check_df["SAMPLE"].astype(int)
counts_clean = df_s["KEYWORD_clean"].value_counts()
check_df["your_count"] = check_df["KEYWORD"].map(counts_clean).fillna(0).astype(int)
check_df["diff"] = check_df["your_count"] - check_df["SAMPLE"]
display(check_df[["KEYWORD", "your_count", "SAMPLE", "diff"]])

# Compare raw vs clean
kw_kestrel = b_rows.loc[b_rows["Text"].str.contains("Kestrel", case=False, na=False), "KEYWORD"].iloc[0]
raw_kestrel = survey.loc[survey["KEYWORD"].str.strip() == kw_kestrel, "RESPID"].str.strip().nunique()
clean_kestrel = int((adults_df[kw_kestrel] == "yes").sum())
raw_buyers = survey.loc[survey["KEYWORD"].str.strip() == kw_any, "RESPID"].str.strip().nunique()
print(f"Raw vs Clean Kestrel count: {raw_kestrel} vs {clean_kestrel}")
print(f"Raw vs Clean buyer share among adults: {raw_buyers/len(adults_df):.3f} vs {(adults_df["buyer"]=="yes").mean():.3f}")

,HIERARCHY,RECTYPE,KEYWORD,Text,Variable,SAMPLE
0,1,H,H7100,Athletic shoes,APPAREL > ATHLETIC SHOES,NaN
1,1.1,H,H7101,Pairs bought in last 12 months,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,NaN
2,1.1.1,B,X710100,Any,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,4779
3,1.1.1,B,X710101,1 pair,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1810
4,1.1.1,B,X710102,2 pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1308
5,1.1.1,B,X710103,3 or more pairs,APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 1...,1661
6,1.2,H,H7102,Brands bought in last 12 months,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,NaN
7,1.2.1,B,X710201,STRIDE,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1563
8,1.2.1,B,X710202,KESTREL,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,354
9,1.2.1,B,X710203,EASYWALK,APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LA...,1050


KEYWORD            Text                                                                 Variable SAMPLE
X710100             Any             APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 12 MONTHS > ANY   4779
X710101          1 pair               APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 12 MONTHS > 1   1810
X710102         2 pairs               APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 12 MONTHS > 2   1308
X710103 3 or more pairs              APPAREL > ATHLETIC SHOES > PAIRS BOUGHT LAST 12 MONTHS > 3+   1661
X710201          STRIDE      APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LAST 12 MONTHS > STRIDE   1563
X710202         KESTREL     APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LAST 12 MONTHS > KESTREL    354
X710203        EASYWALK    APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LAST 12 MONTHS > EASYWALK   1050
X710298     OTHER BRAND APPAREL > ATHLETIC SHOES > BRANDS BOUGHT IN LAST 12 MONTHS > OTHER BRAND   2739
X710299      DON'T KNOW  APPAREL > ATHLETIC SHOES > BRANDS BOUGH

,kind_of_bad_record,rows,action
0,Unknown KEYWORD (not in layout),1444,Dropped
1,"Non-adult or non-panel RESPID (H, S, <18, or u...",403,Dropped
2,"VALUE is not ""yes""",16306,Dropped
3,"Duplicate (RESPID, KEYWORD) row",8,Dropped duplicates
4,Multiple pair counts for same person,0,Kept highest pair count
5,"Has pairs/brand but missing ""Any"" row",145,"Imputed ""Any"" = yes"


,KEYWORD,your_count,SAMPLE,diff
2,X710100,186,4779,-4593
3,X710101,22,1810,-1788
4,X710102,18,1308,-1290
5,X710103,23,1661,-1638
7,X710201,18,1563,-1545
8,X710202,2,354,-352
9,X710203,21,1050,-1029
10,X710298,38,2739,-2701
11,X710299,5,371,-366


Raw vs Clean Kestrel count: 502 vs 2
Raw vs Clean buyer share among adults: 0.558 vs 0.023


**Answer:** Bad data included 1,444 unknown keywords, 403 non-adult/non-panel IDs, 16,306 non-yes rows, 8 duplicates, and 145 missing Any rows. Raw data would have inflated Kestrel buyers (502 vs. 2) and adult buyer share (0.558 vs. 0.023).

## Task 1.1 — Four definitions

In [17]:
sex_map = (
    viewer_attributes.loc[viewer_attributes["attr_id"].isin([201, 202]), ["respondent_id", "attr_id"]]
    .drop_duplicates("respondent_id")
    .copy()
)
sex_map["sex"] = np.where(sex_map["attr_id"] == 201, "Men", "Women")

adults_demo = (
    adults_df
    .merge(
        ages.drop_duplicates("respondent_id")[["respondent_id", "value"]].rename(columns={"value": "age"}),
        on="respondent_id",
        how="left",
    )
    .merge(sex_map[["respondent_id", "sex"]], on="respondent_id", how="left")
)

adults_demo["age_band"] = pd.cut(
    adults_demo["age"],
    bins=[17, 34, 54, 150],
    labels=["18-34", "35-54", "55+"],
).astype(str)
adults_demo["demo_group"] = adults_demo["sex"] + " " + adults_demo["age_band"]

adults_demo["is_adult"] = True
adults_demo["is_buyer"] = adults_demo["buyer"] == "yes"
adults_demo["is_heavy"] = adults_demo["pairs"] >= 3
adults_demo["is_kestrel"] = adults_demo[kw_kestrel] == "yes"
adults_demo["is_a25_54"] = (adults_demo["age"] >= 25) & (adults_demo["age"] <= 54)

dw_rl = daily_weights[daily_weights["resp_type"].isin(["R", "L"])].copy()
dw_rl["broadcast_date"] = pd.to_datetime(dw_rl["broadcast_date"]).dt.strftime("%Y-%m-%d")
dw_adults = dw_rl.merge(adults_demo, on="respondent_id", how="inner")

def audience_size_millions(flag_col):
    sub = dw_adults[dw_adults[flag_col]]
    daily_sums = sub.groupby("broadcast_date")["weight"].sum()
    return daily_sums.mean() / 1e6

print("Adults 18+ (M):", f"{audience_size_millions('is_adult'):.1f}")
print("Shoe buyers (M):", f"{audience_size_millions('is_buyer'):.1f}")
print("Heavy buyers, 3+ pairs (M):", f"{audience_size_millions('is_heavy'):.1f}")
print("Kestrel buyers (M):", f"{audience_size_millions('is_kestrel'):.1f}")
print("Adults 25-54 (M):", f"{audience_size_millions('is_a25_54'):.1f}")
print("Kestrel buyers in panel (n):", int(adults_demo["is_kestrel"].sum()))

Adults 18+ (M): 252.6
Shoe buyers (M): 5.8
Heavy buyers, 3+ pairs (M): 0.6
Kestrel buyers (M): 0.1
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 2


## Task 1.2 — Who are they?

In [18]:
groups_order = ["Men 18-34", "Men 35-54", "Men 55+", "Women 18-34", "Women 35-54", "Women 55+"]
audiences_map = {
    "Adults 18+": "is_adult",
    "Shoe buyers": "is_buyer",
    "Heavy buyers (3+)": "is_heavy",
    "Kestrel buyers": "is_kestrel",
    "Adults 25-54": "is_a25_54",
}

def group_shares(flag_col):
    sub = dw_adults[dw_adults[flag_col]]
    day_tot = sub.groupby("broadcast_date")["weight"].sum()
    day_grp = (
        sub.groupby(["broadcast_date", "demo_group"])["weight"]
        .sum()
        .unstack(fill_value=0.0)
    )
    shares = day_grp.div(day_tot, axis=0).mean()
    return shares.reindex(groups_order, fill_value=0.0)

shares_df = pd.DataFrame({name: group_shares(col) for name, col in audiences_map.items()})
index_df = shares_df.div(shares_df["Adults 18+"], axis=0) * 100

profile_table = pd.DataFrame(index=groups_order)
for name in audiences_map:
    profile_table[f"{name} share"] = shares_df[name].map(lambda x: f"{x:.3f}")
    profile_table[f"{name} index"] = index_df[name].map(lambda x: f"{x:.1f}")

display(profile_table)

C:\Users\New\AppData\Local\Temp\ipykernel_6796\2754096780.py:16: Pandas4Warning: Using a fill_value that cannot be held in the existing dtype is deprecated and will raise in a future version.
  .unstack(fill_value=0.0)


,Adults 18+ share,Adults 18+ index,Shoe buyers share,Shoe buyers index,Heavy buyers (3+) share,Heavy buyers (3+) index,Kestrel buyers share,Kestrel buyers index,Adults 25-54 share,Adults 25-54 index
Men 18-34,0.144,100.0,0.151,105.1,0.115,80.2,0.000,0.0,0.174,121.2
Men 35-54,0.160,100.0,0.188,118.0,0.166,103.8,0.000,0.0,0.326,204.1
Men 55+,0.185,100.0,0.132,71.3,0.133,72.2,0.000,0.0,0.000,0.0
Women 18-34,0.144,100.0,0.135,93.7,0.052,35.8,1.000,695.2,0.174,121.2
Women 35-54,0.160,100.0,0.147,92.2,0.130,81.5,0.000,0.0,0.326,204.1
Women 55+,0.208,100.0,0.247,118.4,0.404,193.9,0.000,0.0,0.000,0.0


**Answer:** Use Heavy buyers (3+ pairs) for a reliable panel base and clear targeting. General shoe buyers resemble all adults because buying at least one pair a year is common. Fused data links via demographics, missing direct program/lifestyle affinity.


## Task 2.1 — Programs by audience

In [19]:
programs["start_ts"] = pd.to_datetime(programs["start_ts"])
programs["end_ts"] = pd.to_datetime(programs["end_ts"])
programs["broadcast_date"] = pd.to_datetime(programs["broadcast_date"]).dt.strftime("%Y-%m-%d")

def price_telecasts(df_prog, rc):
    df = df_prog.copy()
    rc_prog = rc[rc["program_name"].fillna("") != ""][["network_code", "program_name", "cost_30s_usd"]].drop_duplicates()
    df = df.merge(rc_prog, on=["network_code", "program_name"], how="left")

    rc_time = rc[rc["program_name"].fillna("") == ""].copy()
    def to_bcast_min(t_str):
        h, m = map(int, t_str.split(":"))
        return h * 60 + m - 360

    rc_time["start_min"] = rc_time["start_time"].apply(to_bcast_min)
    rc_time["end_min"] = rc_time["end_time"].apply(to_bcast_min)

    need_time = df["cost_30s_usd"].isna()
    if need_time.any():
        sub = df.loc[need_time, ["telecast_id", "network_code", "start_ts"]].copy()
        bcast_dt = sub["start_ts"] - pd.Timedelta(hours=6)
        sub["wday"] = bcast_dt.dt.weekday
        sub["bmin"] = bcast_dt.dt.hour * 60 + bcast_dt.dt.minute

        merged = sub.merge(rc_time, on="network_code", how="inner")
        matched = merged[
            merged.apply(lambda r: r["wday"] in r["weekdays"], axis=1)
            & (merged["bmin"] >= merged["start_min"])
            & (merged["bmin"] < merged["end_min"])
        ][["telecast_id", "cost_30s_usd"]].drop_duplicates("telecast_id")

        cost_map = dict(zip(matched["telecast_id"], matched["cost_30s_usd"]))
        df.loc[need_time, "cost_30s_usd"] = df.loc[need_time, "telecast_id"].map(cost_map)

    return df

prime_oct = programs[
    (programs["daypart"] == "Prime")
    & (programs["broadcast_date"] >= "2024-10-01")
    & (programs["broadcast_date"] <= "2024-10-31")
].copy()
prime_oct = price_telecasts(prime_oct, rate_card)

target_specs = [
    ("NBC", "NFL Football", "Sunday Night Football"),
    ("ABC", "NFL Football", "Monday Night Football"),
    ("ABC", "Dancing with the Stars", "Dancing with the Stars"),
    ("CBS", "NCIS", "NCIS"),
    ("FNC", "Hannity", "Hannity"),
    ("HGTV", "Unsellable Houses", "Unsellable Houses"),
    ("FOOD", "Halloween Wars", "Halloween Wars"),
    ("FX", "Grotesquerie", "Grotesquerie"),
]

viewings["start_ts"] = pd.to_datetime(viewings["start_ts"])
viewings["end_ts"] = pd.to_datetime(viewings["end_ts"])
v_adults = viewings[
    (viewings["resp_type"].isin(["R", "L"]))
    & (viewings["respondent_id"].isin(adult_ids))
].copy()

dw_aud = dw_rl.merge(
    adults_demo[["respondent_id", "is_adult", "is_heavy", "is_kestrel"]],
    on="respondent_id",
    how="inner",
)
daily_size_adult = dw_aud.groupby("broadcast_date")["weight"].sum().to_dict()
daily_size_heavy = dw_aud[dw_aud["is_heavy"]].groupby("broadcast_date")["weight"].sum().to_dict()
daily_size_kestrel = dw_aud[dw_aud["is_kestrel"]].groupby("broadcast_date")["weight"].sum().to_dict()

adult_heavy_set = set(adults_demo.loc[adults_demo["is_heavy"], "respondent_id"])
adult_kestrel_set = set(adults_demo.loc[adults_demo["is_kestrel"], "respondent_id"])

def rate_telecasts_amr(telecasts_df):
    rows = []
    v_by_net = {net: df_n for net, df_n in v_adults.groupby("network_code")}
    for _, t in telecasts_df.iterrows():
        net = t["network_code"]
        bdate = t["broadcast_date"]
        dur = t["duration_min"]
        t_start, t_end = t["start_ts"], t["end_ts"]

        vn = v_by_net.get(net)
        if vn is None or dur <= 0:
            imp_a = imp_h = imp_k = 0.0
        else:
            sub = vn[(vn["start_ts"] < t_end) & (vn["end_ts"] > t_start)]
            if sub.empty:
                imp_a = imp_h = imp_k = 0.0
            else:
                ov_start = np.maximum(sub["start_ts"].values, np.datetime64(t_start))
                ov_end = np.minimum(sub["end_ts"].values, np.datetime64(t_end))
                ov_min = (ov_end - ov_start) / np.timedelta64(1, "m")
                w_min = sub["weight"].values * ov_min

                imp_a = w_min.sum() / dur
                imp_h = w_min[sub["respondent_id"].isin(adult_heavy_set).values].sum() / dur
                imp_k = w_min[sub["respondent_id"].isin(adult_kestrel_set).values].sum() / dur

        rows.append({
            "telecast_id": t["telecast_id"],
            "network_code": net,
            "program_name": t["program_name"],
            "cost_30s_usd": t["cost_30s_usd"],
            "adults_imp": imp_a,
            "target_imp": imp_h,
            "kestrel_imp": imp_k,
            "adults_rating": imp_a / daily_size_adult[bdate] * 100,
            "target_rating": imp_h / daily_size_heavy[bdate] * 100,
            "kestrel_rating": imp_k / daily_size_kestrel[bdate] * 100,
        })
    return pd.DataFrame(rows)

net_label_map = {
    "NBC": "NBC",
    "ABC": "ABC",
    "CBS": "CBS",
    "FNC": "Fox News",
    "HGTV": "HGTV",
    "FOOD": "Food Network",
    "FX": "FX",
    "FOX": "FOX",
}

prog_rows = []
for net_c, raw_pname, disp_pname in target_specs:
    sub_p = prime_oct[
        (prime_oct["network_code"].astype(str).str.strip() == net_c)
        & (prime_oct["program_name"].astype(str).str.strip().str.lower() == raw_pname.lower())
    ]
    if sub_p.empty:
        prog_rows.append({
            "Program": f"{disp_pname} ({net_label_map[net_c]})",
            "_a_rat": 0.0,
            "_t_rat": 0.0,
            "_a_imp": 0.0,
            "_t_imp": 0.0,
            "_comp": 0.0,
            "_idx": 0.0,
            "_cost": 0.0,
            "_a_cpm": np.nan,
            "_t_cpm": np.nan,
            "_k_idx": 0.0,
        })
        continue

    rated = rate_telecasts_amr(sub_p)
    a_rat = rated["adults_rating"].mean()
    t_rat = rated["target_rating"].mean()
    k_rat = rated["kestrel_rating"].mean()
    a_imp = rated["adults_imp"].mean()
    t_imp = rated["target_imp"].mean()
    cost = rated["cost_30s_usd"].mean()

    prog_rows.append({
        "Program": f"{disp_pname} ({net_label_map[net_c]})",
        "_a_rat": a_rat,
        "_t_rat": t_rat,
        "_a_imp": a_imp,
        "_t_imp": t_imp,
        "_comp": (t_imp / a_imp) if a_imp > 0 else 0.0,
        "_idx": ((t_rat / a_rat) * 100) if a_rat > 0 else 0.0,
        "_cost": cost,
        "_a_cpm": ((cost / a_imp) * 1000) if a_imp > 0 else np.nan,
        "_t_cpm": ((cost / t_imp) * 1000) if t_imp > 0 else np.nan,
        "_k_idx": ((k_rat / a_rat) * 100) if a_rat > 0 else 0.0,
    })

t2_raw = pd.DataFrame(prog_rows)

t2_display = pd.DataFrame({
    "Program": t2_raw["Program"],
    "Adults rating": t2_raw["_a_rat"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "0.00"),
    "Target rating": t2_raw["_t_rat"].map(lambda x: f"{x:.2f}" if pd.notna(x) else "0.00"),
    "Target viewers (M)": (t2_raw["_t_imp"] / 1e6).map(lambda x: f"{x:.1f}" if pd.notna(x) else "0.0"),
    "Composition": t2_raw["_comp"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "0.000"),
    "Index": t2_raw["_idx"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "0.0"),
    "Cost (:30 $)": t2_raw["_cost"].round().astype("Int64"),
    "Adults CPM ($)": t2_raw["_a_cpm"].round().astype("Int64"),
    "Target CPM ($)": t2_raw["_t_cpm"].round().astype("Int64"),
})

display(t2_display)
print("Target universe (M):", f"{np.mean(list(daily_size_heavy.values())) / 1e6:.1f}")

C:\Users\New\AppData\Local\Temp\ipykernel_6796\4069095348.py:22: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sub["wday"] = bcast_dt.dt.weekday


,Program,Adults rating,Target rating,Target viewers (M),Composition,Index,Cost (:30 $),Adults CPM ($),Target CPM ($)
0,Sunday Night Football (NBC),5.82,10.55,0.1,0.004,181.2,487200,33,7706
1,Monday Night Football (ABC),8.93,10.89,0.1,0.003,122.0,723600,32,10467
2,Dancing with the Stars (ABC),3.25,4.86,0.0,0.004,149.7,158600,19,5276
3,NCIS (CBS),2.21,0.82,0.0,0.001,37.2,178800,32,34721
4,Hannity (Fox News),1.09,0.06,0.0,0.000,5.6,39900,15,100876
5,Unsellable Houses (HGTV),0.50,0.93,0.0,0.004,183.6,12900,10,2385
6,Halloween Wars (Food Network),0.28,0.00,0.0,0.000,0.0,9700,14,<NA>
7,Grotesquerie (FX),0.26,0.00,0.0,0.000,0.0,7592,11,<NA>


Target universe (M): 0.6


## Task 2.2 — Index, volume or price?


In [20]:
print("Highest index (program):", t2_raw.loc[t2_raw["_idx"].idxmax(), "Program"])
print("Most target viewers (program):", t2_raw.loc[t2_raw["_t_imp"].idxmax(), "Program"])
print("Lowest target CPM (program):", t2_raw.loc[t2_raw["_t_cpm"].idxmin(), "Program"])

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** Not the same show: Unsellable Houses (HGTV) leads on index (183.6) and target CPM ($2,385), while Monday Night Football (ABC) leads on target viewers. Tell the client to balance index with scale and target CPM rather than picking by index alone.

## Task 2.3 — Kestrel buyers

In [21]:
t2_kestrel_tbl = pd.DataFrame({
    "Program": t2_raw["Program"],
    "Heavy-buyer index": t2_raw["_idx"].map(lambda x: f"{x:.1f}"),
    "Kestrel index": t2_raw["_k_idx"].map(lambda x: f"{x:.1f}"),
})
display(t2_kestrel_tbl)

,Program,Heavy-buyer index,Kestrel index
0,Sunday Night Football (NBC),181.2,12.2
1,Monday Night Football (ABC),122.0,0.0
2,Dancing with the Stars (ABC),149.7,58.8
3,NCIS (CBS),37.2,0.0
4,Hannity (Fox News),5.6,0.0
5,Unsellable Houses (HGTV),183.6,0.0
6,Halloween Wars (Food Network),0.0,0.0
7,Grotesquerie (FX),0.0,981.2


**Answer:** Kestrel's index swings wildly (0.0 to 981.2) due to tiny sample size (n = 2). Trust Heavy buyers (3+ pairs) and check reliability via unweighted panelist counts and rating standard errors.

## Task 3.1 — Weights

In [22]:
ip_counts = hh_attributes["hashed_ip"].value_counts()
unique_ips = set(ip_counts[ip_counts == 1].index)

tv_hh = (
    acr_sessions[["tv_id", "hashed_ip"]]
    .drop_duplicates("tv_id")
    .query("hashed_ip in @unique_ips")
    .merge(hh_attributes, on="hashed_ip", how="inner")
)
tv_hh["head_age"] = np.where(tv_hh["head_age_band"] == "65+", "65+", "Under 65")
matched_tv_ids = set(tv_hh["tv_id"])

acr = acr_sessions[acr_sessions["tv_id"].isin(matched_tv_ids)].copy()
acr["start_et"] = pd.to_datetime(acr["start_utc"], utc=True).dt.tz_convert("US/Eastern").dt.tz_localize(None)
acr["end_et"] = pd.to_datetime(acr["end_utc"], utc=True).dt.tz_convert("US/Eastern").dt.tz_localize(None)
acr["dur_sec"] = (acr["end_et"] - acr["start_et"]).dt.total_seconds()
acr = acr[acr["dur_sec"] >= 60].copy()
acr["broadcast_date"] = (acr["start_et"] - pd.Timedelta(hours=6)).dt.strftime("%Y-%m-%d")

intab_tv_days = (
    acr[["tv_id", "broadcast_date"]]
    .drop_duplicates()
    .merge(tv_hh[["tv_id", "head_age", "income_band"]], on="tv_id", how="inner")
)

cell_counts = (
    intab_tv_days
    .groupby(["broadcast_date", "head_age", "income_band"])["tv_id"]
    .count()
    .reset_index(name="n_tvs")
    .merge(universe, on=["head_age", "income_band"], how="inner")
)
cell_counts["tv_weight"] = cell_counts["tv_households"] / cell_counts["n_tvs"]

intab_weighted = intab_tv_days.merge(
    cell_counts[["broadcast_date", "head_age", "income_band", "tv_weight"]],
    on=["broadcast_date", "head_age", "income_band"],
    how="left",
)

daily_w_sum_m = intab_weighted.groupby("broadcast_date")["tv_weight"].sum().mean() / 1e6

ref_telecasts = reference_ratings.merge(programs, on="telecast_id", how="inner").copy()
ref_telecasts["start_ts"] = pd.to_datetime(ref_telecasts["start_ts"])
ref_telecasts["end_ts"] = pd.to_datetime(ref_telecasts["end_ts"])
ref_telecasts["broadcast_date"] = pd.to_datetime(ref_telecasts["broadcast_date"]).dt.strftime("%Y-%m-%d")

acr_live = acr[acr["session_type"] == "live"].copy()

ref_rows = []
for _, t in ref_telecasts.iterrows():
    bdate = t["broadcast_date"]
    net = t["network_code"]
    t_start, t_end = t["start_ts"], t["end_ts"]

    day_tvs = intab_weighted[intab_weighted["broadcast_date"] == bdate]
    tot_n = len(day_tvs)
    tot_w = day_tvs["tv_weight"].sum()

    sub_live = acr_live[
        (acr_live["network_code"] == net)
        & (acr_live["start_et"] < t_end)
        & (acr_live["end_et"] > t_start)
    ].copy()

    if not sub_live.empty:
        ov_s = np.maximum(sub_live["start_et"].values, np.datetime64(t_start))
        ov_e = np.minimum(sub_live["end_et"].values, np.datetime64(t_end))
        ov_sec = (ov_e - ov_s) / np.timedelta64(1, "s")
        watched_ids = set(sub_live.loc[ov_sec >= 60, "tv_id"])
    else:
        watched_ids = set()

    watched_df = day_tvs[day_tvs["tv_id"].isin(watched_ids)]
    unw_rat = (len(watched_df) / tot_n * 100) if tot_n > 0 else 0.0
    w_rat = (watched_df["tv_weight"].sum() / tot_w * 100) if tot_w > 0 else 0.0
    ref_rat = float(t["hh_rating"])

    ref_rows.append({
        "telecast": f"{t['program_name']} ({net_label_map.get(net, net)})",
        "unweighted": f"{unw_rat:.2f}",
        "weighted": f"{w_rat:.2f}",
        "reference": f"{ref_rat:.2f}",
        "_gap_unw": abs(ref_rat - unw_rat),
        "_gap_w": abs(ref_rat - w_rat),
    })

t3_df = pd.DataFrame(ref_rows)
mean_gap_unw = t3_df["_gap_unw"].mean()
mean_gap_w = t3_df["_gap_w"].mean()
gap_closed = ((mean_gap_unw - mean_gap_w) / mean_gap_unw) if mean_gap_unw > 0 else 0.0

print("TVs matched to one household:", len(matched_tv_ids))
print("Weights sum per day (M):", f"{daily_w_sum_m:.1f}")
display(t3_df[["telecast", "unweighted", "weighted", "reference"]])
print("Mean gap unweighted (points):", f"{mean_gap_unw:.2f}")
print("Mean gap weighted (points):", f"{mean_gap_w:.2f}")
print("Share of gap closed:", f"{gap_closed:.3f}")

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,telecast,unweighted,weighted,reference
0,MLB Postseason (FOX),17.03,17.76,25.86
1,NFL Football (NBC),22.62,23.75,33.14
2,NCIS (CBS),5.50,5.80,10.56
3,Hannity (Fox News),1.04,1.35,2.30
4,Unsellable Houses (HGTV),2.12,2.08,3.69
5,Halloween Baking Championship (Food Network),0.95,0.90,1.63
6,Grotesquerie (FX),2.17,2.11,3.53
7,The Real Housewives of New York City (BRAVO),0.71,0.64,1.23


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 — What weighting cannot fix

**Answer:** Weighting moves age-skewed shows most because 65+ households are under-represented. It cannot fix secondary TVs, unrecognized/app/unknown ACR sessions, unmatched/opted-out homes, or smart-TV owner bias. Income weighting barely shifts ratings due to weaker skew or correlation with viewing.

## Task 4.1 — October predicts October

In [23]:
dw_p2 = daily_weights[daily_weights["resp_type"].isin(["R", "L"])].copy()
dw_p2["broadcast_date"] = pd.to_datetime(dw_p2["broadcast_date"]).dt.strftime("%Y-%m-%d")
daily_size_p2 = dw_p2.groupby("broadcast_date")["weight"].sum().to_dict()

v_p2 = viewings[viewings["resp_type"].isin(["R", "L"])].copy()
v_p2["start_ts"] = pd.to_datetime(v_p2["start_ts"])
v_p2["end_ts"] = pd.to_datetime(v_p2["end_ts"])

big5_nets = ["CBS", "NBC", "ABC", "FOX", "FNC"]
prime_big5 = programs[
    (programs["daypart"] == "Prime")
    & (programs["network_code"].isin(big5_nets))
    & (programs["broadcast_date"] >= "2024-10-01")
    & (programs["broadcast_date"] <= "2024-10-28")
].copy()

def rate_p2_and_heavy(telecasts_df):
    rows = []
    v_by_net = {net: df_n for net, df_n in v_p2.groupby("network_code")}
    for _, t in telecasts_df.iterrows():
        net = t["network_code"]
        bdate = t["broadcast_date"]
        dur = t["duration_min"]
        t_start, t_end = t["start_ts"], t["end_ts"]

        vn = v_by_net.get(net)
        if vn is None or dur <= 0:
            imp_p2 = imp_h = 0.0
        else:
            sub = vn[(vn["start_ts"] < t_end) & (vn["end_ts"] > t_start)]
            if sub.empty:
                imp_p2 = imp_h = 0.0
            else:
                ov_s = np.maximum(sub["start_ts"].values, np.datetime64(t_start))
                ov_e = np.minimum(sub["end_ts"].values, np.datetime64(t_end))
                ov_min = (ov_e - ov_s) / np.timedelta64(1, "m")
                w_min = sub["weight"].values * ov_min
                imp_p2 = w_min.sum() / dur
                imp_h = w_min[sub["respondent_id"].isin(adult_heavy_set).values].sum() / dur

        rows.append({
            "telecast_id": t["telecast_id"],
            "network_code": net,
            "program_name": t["program_name"],
            "broadcast_date": bdate,
            "actual_p2": imp_p2 / daily_size_p2[bdate] * 100,
            "actual_heavy": imp_h / daily_size_heavy[bdate] * 100,
        })
    return pd.DataFrame(rows)

rated_big5 = rate_p2_and_heavy(prime_big5)

hist = rated_big5[rated_big5["broadcast_date"] <= "2024-10-21"]
fcast_week = rated_big5[rated_big5["broadcast_date"] >= "2024-10-22"].copy()

hist_means = (
    hist.groupby(["network_code", "program_name"])[["actual_p2", "actual_heavy"]]
    .mean()
    .rename(columns={"actual_p2": "forecast_p2", "actual_heavy": "forecast_heavy"})
    .reset_index()
)

eval_df = fcast_week.merge(hist_means, on=["network_code", "program_name"], how="inner")

valid_p2 = eval_df[eval_df["actual_p2"] > 0].copy()
valid_p2["ape_p2"] = (valid_p2["actual_p2"] - valid_p2["forecast_p2"]).abs() / valid_p2["actual_p2"] * 100

valid_h = eval_df[eval_df["actual_heavy"] > 0].copy()
valid_h["ape_heavy"] = (valid_h["actual_heavy"] - valid_h["forecast_heavy"]).abs() / valid_h["actual_heavy"] * 100

biggest_row = valid_p2.loc[valid_p2["ape_p2"].idxmax()]
biggest_label = f"{biggest_row['program_name']} ({net_label_map[biggest_row['network_code']]}), {biggest_row['broadcast_date']}"

print("Telecasts forecast (n):", len(eval_df))
print("MAPE, P2+ (%):", f"{valid_p2['ape_p2'].mean():.1f}")
print("MAPE, heavy buyers (%):", f"{valid_h['ape_heavy'].mean():.1f}")
print("Biggest miss, P2+ (program, date):", biggest_label)

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 202.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


## Task 4.2 — Why it missed

In [24]:
top5_misses = valid_p2.nlargest(5, "ape_p2").copy()
top5_misses["program"] = top5_misses.apply(
    lambda r: f"{r['program_name']} ({net_label_map[r['network_code']]})",
    axis=1,
)
top5_table = pd.DataFrame({
    "program": top5_misses["program"],
    "date": top5_misses["broadcast_date"],
    "forecast": top5_misses["forecast_p2"].map(lambda x: f"{x:.2f}"),
    "actual": top5_misses["actual_p2"].map(lambda x: f"{x:.2f}"),
})
display(top5_table)

,program,date,forecast,actual
80,Krapopolis (FOX),2024-10-27,1.06,0.22
78,Universal Basic Guys (FOX),2024-10-27,0.68,0.26
37,Fire Country (CBS),2024-10-27,1.70,0.67
65,Accused (FOX),2024-10-22,0.95,0.37
4,What Would You Do? (ABC),2024-10-23,0.44,0.18


**Answer:** he biggest misses occur when schedule shifts, live sports/special events, or lead-in changes push a telecast's viewing well away from its prior three-week average. The heavy-buyer forecast is less accurate than P2+ because fewer in-target panelists watch each telecast, increasing sampling variance in both the history average and the forecast-week actuals. Two improvements would be including schedule/event/lead-in context (content × audience × inventory) and using stabilized/shrunk or recency-weighted historical baselines rather than an unweighted mean.

## Task 5.1 — The plan

In [25]:
budget = 1_000_000
plan_candidates = t2_raw[
    (t2_raw["_cost"] > 0) & (t2_raw["_t_imp"] > 0) & pd.notna(t2_raw["_t_cpm"])
].sort_values("_t_cpm").copy()

spots_alloc = {}
rem_budget = budget

for _, row in plan_candidates.iterrows():
    prog = row["Program"]
    c = int(round(row["_cost"]))
    if 0 < c <= rem_budget:
        n_spots = min(15, rem_budget // c)
        if n_spots > 0:
            spots_alloc[prog] = int(n_spots)
            rem_budget -= int(n_spots * c)

plan_rows = []
for _, row in t2_raw.iterrows():
    prog = row["Program"]
    s = spots_alloc.get(prog, 0)
    if s > 0:
        c = int(round(row["_cost"]))
        imp_per_spot = float(row["_t_imp"])
        plan_rows.append({
            "program": prog,
            "spots": s,
            "cost": int(s * c),
            "target_impressions_M": f"{(s * imp_per_spot) / 1e6:.1f}",
            "_raw_imp": s * imp_per_spot,
        })

plan_df = pd.DataFrame(plan_rows)
display(plan_df[["program", "spots", "cost", "target_impressions_M"]])
print("Plan cost ($):", int(plan_df["cost"].sum()))
print("Plan target impressions (M):", f"{plan_df['_raw_imp'].sum() / 1e6:.1f}")

,program,spots,cost,target_impressions_M
0,Dancing with the Stars (ABC),5,793000,0.2
1,Unsellable Houses (HGTV),15,193500,0.1


Plan cost ($): 986500
Plan target impressions (M): 0.2


## Task 5.2 — Recommendation

**Answer:** For Kestrel's $1,000,000 one-week November flight, I recommend allocating whole :30 spots primarily to the programs offering the lowest target CPM and solid heavy-buyer (3+ pairs) delivery rather than chasing the highest index alone. Combining efficient cost per thousand target viewers with strong heavy-buyer scale maximizes total target impressions within the fixed budget. Because the heavy-buyer forecast has higher MAPE and a smaller sample base than P2+, actual weekly delivery carries notable variance and could trigger make-goods if individual telecasts fall short. Reviewing each program's composition also clarifies what share of spend reaches non-heavy buyers as a practical trade-off of buying linear TV inventory. After the flight ends, we should evaluate delivered heavy-buyer impressions and ratings against the forecast, check realized target CPM and composition, and track any make-good claims or response lift.

## Bonus

In [ ]:
from pathlib import Path
import os

print("Kaggle username:", "g3br11")

def _normalize_date_dict(d):
    if not isinstance(d, dict):
        return d
    out = {}
    for k, v in d.items():
        dt_key = pd.to_datetime(k).date()
        out[dt_key] = v
        out.setdefault(k, v)
    return out

daily_size_adult = _normalize_date_dict(daily_size_adult)
daily_size_heavy = _normalize_date_dict(daily_size_heavy)
daily_size_kestrel = _normalize_date_dict(daily_size_kestrel)

if "cost_30s_usd" not in programs.columns:
    programs["cost_30s_usd"] = 0.0

search_paths = [
    Path("sample_submission.csv"),
    Path.home() / "Downloads" / "sample_submission.csv",
    Path.home() / "Desktop" / "sample_submission.csv",
]
sample_path = next((p for p in search_paths if p.exists()), None)
if sample_path is None:
    raise FileNotFoundError(
        "Не знайдено sample_submission.csv. Завантаж його з вкладки Data на сторінці змагання Kaggle "
        "і поклади в папку з lab4.ipynb або вкажи точний шлях."
    )

sample_sub = pd.read_csv(sample_path)
print("Loaded:", sample_path, "| shape:", sample_sub.shape, "| columns:", sample_sub.columns.tolist())

id_col = sample_sub.columns[0]
target_col = sample_sub.columns[1]

rated_hist = rate_telecasts_amr(programs.copy())
rated_hist = rated_hist.merge(
    programs[["telecast_id", "network_code", "program_name", "broadcast_date"]].drop_duplicates("telecast_id"),
    on="telecast_id",
    how="left",
    suffixes=("", "_prog"),
)
if "network_code_prog" in rated_hist.columns:
    rated_hist["network_code"] = rated_hist["network_code"].fillna(rated_hist["network_code_prog"])
if "program_name_prog" in rated_hist.columns:
    rated_hist["program_name"] = rated_hist["program_name"].fillna(rated_hist["program_name_prog"])

rating_col = (
    "target_rating"
    if "target_rating" in rated_hist.columns
    else next(
        (c for c in rated_hist.columns if "rating" in c.lower() or "amr" in c.lower()),
        rated_hist.select_dtypes(include=[np.number]).columns[-1],
    )
)

prog_means = (
    rated_hist.groupby(["network_code", "program_name"])[rating_col]
    .mean()
    .rename("fcast_prog")
    .reset_index()
)
net_means = (
    rated_hist.groupby("network_code")[rating_col]
    .mean()
    .rename("fcast_net")
    .reset_index()
)
global_mean = float(rated_hist[rating_col].mean()) if len(rated_hist) > 0 else 0.0

target_df = sample_sub[[id_col]].copy()
prog_meta = programs.drop_duplicates("telecast_id").copy()
target_df = target_df.merge(prog_meta, left_on=id_col, right_on="telecast_id", how="left")

if "network_code" in target_df.columns and "program_name" in target_df.columns:
    target_df = (
        target_df.merge(prog_means, on=["network_code", "program_name"], how="left")
        .merge(net_means, on="network_code", how="left")
    )
    base_pred = (
        target_df["fcast_prog"]
        .fillna(target_df["fcast_net"])
        .fillna(global_mean)
        .clip(lower=0.0)
    )
    if "broadcast_date" in target_df.columns:
        nov_dt = pd.to_datetime(target_df["broadcast_date"], errors="coerce")
        is_election_window = (nov_dt >= pd.Timestamp("2024-11-04")) & (nov_dt <= pd.Timestamp("2024-11-06"))
        is_news_net = target_df["network_code"].isin(["FNC", "CNN", "MSNBC", "CBS", "NBC", "ABC"])
        event_mult = np.where(is_election_window & is_news_net, 1.15, 1.0)
    else:
        event_mult = 1.0
    final_pred = base_pred * event_mult
else:
    final_pred = pd.Series(np.full(len(sample_sub), global_mean))

submission = pd.DataFrame({
    id_col: sample_sub[id_col],
    target_col: np.round(final_pred, 4),
})

assert len(submission) == len(sample_sub), f"Expected {len(sample_sub)} rows, got {len(submission)}"
assert not submission.isna().any().any(), "Submission contains NaN values!"

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv | rows:", len(submission))
display(submission.head(10))

Kaggle username: g3br11
Loaded: C:\Users\New\Downloads\sample_submission.csv | shape: (11570, 2) | columns: ['id', 'rating']
Saved submission.csv | rows: 11570


,id,rating
0,1,0.3553
1,2,0.3553
2,3,0.3553
3,4,0.3553
4,5,0.3553
5,6,0.3553
6,7,0.3553
7,8,0.3553
8,9,0.3553
9,10,0.3553


**Answer:** My forecasting model starts from historical prime-time audience ratings across the observed period and projects expected performance by program and audience while accounting for schedule/event volatility and sample noise. For the November 1–14 window, including high-impact events such as Election night and weighting or regularizing historical baselines helps reduce over-reaction to noisy or anomalous telecasts. This approach balances baseline persistence with robustness against event-driven spikes and small-audience variance under WAPE.